# Adulteration detection code

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load dataset
df = pd.read_csv("grain_features_Basmati_corrected.csv")

pb1121 = df[df["class"]=="PB 1121"]["aspect_ratio"]
pr14 = df[df["class"]=="PR 14"]["aspect_ratio"]

levels = [1,2,5,10,15,20]
simulations = 100

mean_p = []
std_p = []

for level in levels:
    
    p_values = []
    
    for i in range(simulations):
        
        n_pr14 = int(len(pb1121)*level/100)
        n_pb = len(pb1121) - n_pr14
        
        mixture = np.concatenate([
            pb1121.sample(n_pb, replace=True),
            pr14.sample(n_pr14, replace=True)
        ])
        
        stat,p = ttest_ind(pb1121, mixture)
        p_values.append(p)
    
    mean_p.append(np.mean(p_values))
    std_p.append(np.std(p_values))

# Plot
plt.figure(figsize=(8,6))

plt.errorbar(
    levels,
    mean_p,
    yerr=std_p,
    marker='o',
    capsize=5
)

plt.axhline(0.05, linestyle='--', label='Significance threshold')

plt.xlabel("Adulteration Level (%)", fontsize=18, fontweight="bold")
plt.ylabel("Mean p-value", fontsize=18, fontweight="bold")
plt.xticks(fontsize=14, fontweight="bold")
plt.yticks(fontsize=14, fontweight="bold")
#plt.title("Minimum Detectable Adulteration Level (Monte Carlo Simulation)")
plt.legend()

plt.grid(alpha=0.3)

plt.savefig("MDAL_monte_carlo.png",dpi=600)
plt.savefig("MDAL_monte_carlo.pdf",dpi=600)

plt.show()